# 4.4 Subsection

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.ndimage import label

def ice_route_for_date_and_threshold(target_date_str, thr_ice):
    """
    Compute ice-safe fraction, components, and route metrics
    for a given date and SIC threshold.

    Returns dict with:
      - date
      - thr_pct
      - ice_safe_pct_of_sea
      - n_components
      - route_exists
      - route_nm
      - ratio
    """
    target_dt = np.datetime64(target_date_str)
    time_vals = ds_ice["time"].values

    # nearest time index
    idx_nearest = int(np.argmin(np.abs(time_vals - target_dt)))
    t_sel = time_vals[idx_nearest]

    # 2D SIC slice at that time
    siconc_t = ds_ice["siconc"].isel(time=idx_nearest)  # [lat, lon]

    # Regrid to routing grid
    siconc_rg = siconc_t.interp(
        latitude=("latitude", lat_rg),
        longitude=("longitude", lon_rg),
        method="linear"
    ).values  # (n_lat, n_lon)

    # Treat NaNs as ice-free
    nan_frac = np.isnan(siconc_rg).mean()
    print(f"{target_date_str}, thr={thr_ice:.2f}: NaN fraction={nan_frac:.3f}")
    siconc_rg = np.where(np.isnan(siconc_rg), 0.0, siconc_rg)

    # Ice-safe mask = sea & SIC < threshold
    ice_safe_mask = np.logical_and(sea_mask_rg, siconc_rg < thr_ice)

    sea_fraction = sea_mask_rg.mean()
    ice_safe_fraction = ice_safe_mask.mean()
    ice_safe_pct_of_sea = 100.0 * ice_safe_fraction / sea_fraction

    labels, ncomp = label(
        ice_safe_mask.astype(int),
        structure=np.ones((3, 3), int)
    )

    start_label = labels[start_idx]
    goal_label  = labels[goal_idx]
    route_exists = (start_label > 0) and (start_label == goal_label)

    route_nm = np.nan
    ratio = np.nan

    if route_exists:
        path_idx = astar_route(
            ice_safe_mask,
            start_idx,
            goal_idx,
            lat_rg_2d,
            lon_rg_2d,
        )
        route_lat = np.array([lat_rg[i] for i, j in path_idx])
        route_lon = np.array([lon_rg[j] for i, j in path_idx])

        dist_cum_nm = 0.0
        for k in range(1, len(route_lat)):
            dist_cum_nm += haversine_nm(
                route_lat[k-1], route_lon[k-1],
                route_lat[k],   route_lon[k],
            )
        route_nm = dist_cum_nm
        ratio = route_nm / GC_NM

    print(
        f"  ice-safe={ice_safe_pct_of_sea:5.1f}% of sea, "
        f"components={ncomp}, route={route_exists}, "
        f"route_nm={route_nm if route_exists else np.nan:.1f}, "
        f"ratio={ratio if route_exists else np.nan:.3f}"
    )

    return dict(
        date=np.datetime64(target_date_str),
        thr_pct=100.0 * thr_ice,
        ice_safe_pct_of_sea=ice_safe_pct_of_sea,
        n_components=int(ncomp),
        route_exists=bool(route_exists),
        route_nm=float(route_nm) if route_exists else np.nan,
        ratio=float(ratio) if route_exists else np.nan,
    )